## Step 0: Setup & Config

In [ ]:
import json
import re
import string
import requests
import os
from datetime import datetime
from tqdm.auto import tqdm

# === CONFIGURATION ===
LLAMA_SERVER_URL = "http://localhost:8080/v1/chat/completions"

INPUT_FILE = "../output/eval/just_combined.json"
OUTPUT_DIR = "../output/eval/9b"

In [ ]:
from openai import OpenAI

client = OpenAI(
    base_url="http://localhost:8080/v1",
    api_key="lm-studio"  # OpenAI SDK requires a key, but it accepts anything
)

## Step 1: Load Data

In [3]:
with open(INPUT_FILE) as f:
    samples = json.load(f)

amb = [s for s in samples if s["is_ambiguous"]]
unamb = [s for s in samples if not s["is_ambiguous"]]
print(f"Loaded {len(samples)} samples: {len(amb)} ambiguous, {len(unamb)} unambiguous")

Loaded 400 samples: 200 ambiguous, 200 unambiguous


## Generate Oracle Q&A (Qwen 3.5 9B)

In [ ]:
ORACLE_SYSTEM_PROMPT = f"""You are an expert at generating clarifying questions for ambiguous queries.

Given an ambiguous question and a list of its possible intended interpretations, you must:
1. Write a single, concise clarifying question (under 50 words) that distinguishes between the interpretations
2. Write one clarification response per interpretation - a short phrase the user would say to indicate which interpretation they meant

You MUST follow this exact output format with no deviation:
Clarification Question: <your question here>
Clarification Response 1: <response for interpretation 1>
Clarification Response 2: <response for interpretation 2>
(one line per interpretation, numbered sequentially)

Example 1:
Ambiguous Question: Who has the highest goals in world football?
Intended Interpretation 1: Who has the highest goals in men's world international football?
Intended Interpretation 2: Who has the highest goals all-time in men's football?
Intended Interpretation 3: Who has the highest goals in women's world international football?

Clarification Question: Are you referring to the highest goals in men's world international football, or women's?
Clarification Response 1: The highest goals in men's world international football.
Clarification Response 2: The highest goals all-time in men's football.
Clarification Response 3: The highest goals in women's world international football.

Example 2:
Ambiguous Question: Who won the last olympic men's hockey?
Intended Interpretation 1: Who won Olympic men's ice hockey in 2014?
Intended Interpretation 2: Who won Olympic men's ice hockey in 2010?
Intended Interpretation 3: Who won Olympic men's ice hockey in 2006?
Intended Interpretation 4: Who won the 2016 olympic men's field hockey?
Intended Interpretation 5: Who won the 2012 olympic men's field hockey?
Intended Interpretation 6: Who won the 2008 olympic men's field hockey?

Clarification Question: Which year? Are you referring to field hockey or ice hockey?
Clarification Response 1: 2014, ice hockey.
Clarification Response 2: 2010, ice hockey.
Clarification Response 3: 2006, ice hockey.
Clarification Response 4: 2016, field hockey.
Clarification Response 5: 2012, field hockey.
Clarification Response 6: 2008, field hockey.

Rules:
- The clarification question must differentiate ALL interpretations, in a concise question
- Each clarification response should be a brief, natural user reply (a few words to a short phrase)
- Do not add explanations, preamble, or any extra text outside the format
"""

assert ORACLE_SYSTEM_PROMPT, "Needs to be valid"

def call_llm(prompt_text):
    """Call LM Studio using the Python OpenAI SDK."""
    response = client.chat.completions.create(
        model="local-model", # Replace with your LM Studio model identifier if needed
        messages=[
            {"role": "system", "content": ORACLE_SYSTEM_PROMPT},
            {"role": "user", "content": prompt_text}
            ],
        temperature=0.0,
        max_tokens=500
    )
    return response.choices[0].message.content.strip()

def normalize_text(s):
    """Lower text, remove articles, punctuation, and extra whitespace."""
    s = s.lower()
    s = re.sub(r'\b(a|an|the)\b', ' ', s)
    s = s.translate(str.maketrans('', '', string.punctuation))
    s = ' '.join(s.split())
    return s

def build_oracle_prompt(input_x, all_interpretations):
    interp_lines = []
    for i, interp in enumerate(all_interpretations):
        interp_lines.append(f"Intended Interpretation {i+1}: {interp['disambiguated_question']}")
    interp_block = "\n".join(interp_lines)

    return f"""Given the Ambiguous Question and several possible Intended Interpretations, ask a Clarification Question and provide Clarification Responses corresponding to each Intended Interpretation respectively.
Ambiguous Question: {input_x}
{interp_block}"""

def parse_oracle_response(response_text, num_interpretations):
    lines = response_text.strip().split("\n")
    oracle_clarifying_question = None
    oracle_clarifying_responses = []

    for line in lines:
        line = line.strip()
        if line.startswith("Clarification Question:"):
            oracle_clarifying_question = line[len("Clarification Question:"):].strip()
        for i in range(1, num_interpretations + 1):
            prefix = f"Clarification Response {i}:"
            if line.startswith(prefix):
                oracle_clarifying_responses.append(line[len(prefix):].strip())

    return oracle_clarifying_question, oracle_clarifying_responses

def generate_oracle_clarifications(samples):
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_file = f"{OUTPUT_DIR}/reworked_4b_oracle_log_{timestamp}.jsonl"

    with open(log_file, "w", encoding="utf-8") as log_f:
        for entry in tqdm(samples, desc="Generating Oracle Q&A (Qwen 3.5 9B)"):
            if not entry["is_ambiguous"]:
                continue

            input_x = entry["input_x"]
            all_interpretations = entry["all_interpretations"]

            # Deduplicate interpretations
            seen = {}
            unique_interps = []
            for interp in all_interpretations:
                key = normalize_text(interp["disambiguated_question"])
                if key in seen:
                    existing = unique_interps[seen[key]]
                    for ans in interp["answer"]:
                        if normalize_text(ans) not in [normalize_text(a) for a in existing["answer"]]:
                            existing["answer"].append(ans)
                else:
                    seen[key] = len(unique_interps)
                    unique_interps.append({
                        "disambiguated_question": interp["disambiguated_question"],
                        "answer": list(interp["answer"])
                    })

            prompt_text = build_oracle_prompt(input_x, unique_interps)

            try:
                raw_output = call_llm(prompt_text)
                print("\n\n\n-----------------------------------------------------")
                print(raw_output)
                print("-----------------------------------------------------")
                oracle_question, oracle_responses = parse_oracle_response(raw_output, len(unique_interps))

                entry["oracle_clarifying_question"] = oracle_question
                entry["oracle_clarifying_responses"] = []

                for idx, interp in enumerate(unique_interps):
                    response_text = oracle_responses[idx] if idx < len(oracle_responses) else None
                    entry["oracle_clarifying_responses"].append({
                        "clarification_response": response_text,
                        "disambiguated_question": interp["disambiguated_question"],
                        "answer": interp["answer"]
                    })

                log_entry = {
                    "id": entry["id"],
                    "input_x": input_x,
                    "raw_oracle": raw_output,
                    "oracle_clarifying_question": oracle_question,
                    "oracle_clarifying_responses": entry["oracle_clarifying_responses"]
                }
                log_f.write(json.dumps(log_entry) + "\n")
                log_f.flush()

            except Exception as e:
                print(f"Error for '{input_x}': {e}")
                continue

## Visualise Some Examples

In [ ]:
import json
import re
import string

def visualize_oracle_prompts(samples, num_to_show=3):
    count = 0
    for entry in samples:
        if not entry["is_ambiguous"]:
            continue
        
        if count >= num_to_show:
            break
            
        input_x = entry["input_x"]
        all_interpretations = entry["all_interpretations"]

        # Deduplicate interpretations
        seen = {}
        unique_interps = []
        for interp in all_interpretations:
            key = normalize_text(interp["disambiguated_question"])
            if key in seen:
                existing = unique_interps[seen[key]]
                for ans in interp["answer"]:
                    if normalize_text(ans) not in [normalize_text(a) for a in existing["answer"]]:
                        existing["answer"].append(ans)
            else:
                seen[key] = len(unique_interps)
                unique_interps.append({
                    "disambiguated_question": interp["disambiguated_question"],
                    "answer": list(interp["answer"])
                })

        prompt_text = build_oracle_prompt(input_x, unique_interps)
        
        print(f"{'='*80}")
        print(f"SAMPLE ID: {entry.get('id', 'N/A')}")
        print(f"AMBIGUOUS QUESTION: {input_x}")
        print(f"{'-'*40}")
        print("GENERATED PROMPT:")
        print(prompt_text)
        print(f"{'='*80}\n")
        
        count += 1


In [ ]:
visualize_oracle_prompts(samples, num_to_show=2)

In [ ]:
generate_oracle_clarifications(samples)

# Verify
has_oracle = sum(1 for s in samples if s["is_ambiguous"] and "oracle_clarifying_question" in s and s["oracle_clarifying_question"])
print(f"\nOracle Q&A generated for {has_oracle}/{len(amb)} ambiguous samples")

# Spot check
for s in samples:
    if s["is_ambiguous"] and "oracle_clarifying_question" in s:
        print(f"\nQ: {s['input_x']}")
        print(f"Oracle Q: {s['oracle_clarifying_question']}")
        for r in s["oracle_clarifying_responses"][:3]:
            print(f"  -> {r['clarification_response']}  (gold: {r['answer']})")
        break

Save the oracle answers into a new dataset

In [ ]:

for sample in samples:
    # We only care about modifying the ambiguous samples
    if not sample["is_ambiguous"]:
        continue
    print(sample)
    break


with open("../eval/oracle_9b.json", 'w', encoding='utf-8') as f:
    json.dump(samples, f, indent=4)

{'id': '-307381104523119272', 'input_x': 'Who dies at the end of the movie remember the titans?', 'is_ambiguous': True, 'all_interpretations': [{'disambiguated_question': 'Who is the character that dies at the end of the movie remember the titans?', 'answer': ['Bertier', 'Gerry Bertier']}, {'disambiguated_question': 'Who is the actor of the character that dies at the end of the movie remember the titans?', 'answer': ['Ryan Hurst', 'Hurst']}, {'disambiguated_question': 'Who is the character that dies at the end of the movie remember the titans?', 'answer': ['Bertier', 'Gerry Bertier']}, {'disambiguated_question': 'Who is the actor of the character that dies at the end of the movie remember the titans?', 'answer': ['Ryan Hurst', 'Hurst']}], 'gold_intent_question': 'Who is the character that dies at the end of the movie remember the titans?', 'gold_output_y_star': ['Bertier'], 'oracle_clarifying_question': 'Are you asking for the name of the character who dies, or the name of the actor wh

In [ ]:
with open("../output/eval/oracle_9b.json") as f:
    samples = json.load(f)

In [6]:
samples[0]

{'id': '-7729753255484758871',
 'input_x': 'Who did america fight during world war 1?',
 'is_ambiguous': False,
 'gold_output_y_star': ['Germany', 'the Austro - Hungarian Empire']}